In [ ]:
#task1 A

In [ ]:
from pathlib import Path
import os

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import librosa
import librosa.display

In [ ]:
DATASET_PATH = Path(r"D:\audiopro\ravdess")

wav_files = list(DATASET_PATH.rglob("*.wav"))

print("Dataset exists:", DATASET_PATH.exists())
print("Total WAV files found:", len(wav_files))



In [ ]:
def parse_ravdess_filename(path: str) -> dict:
    filename = Path(path).name
    parts = filename.replace(".wav", "").split("-")

    modality_map = {
        "01": "full-AV",
        "02": "video-only",
        "03": "audio-only"
    }

    vocal_channel_map = {
        "01": "speech",
        "02": "song"
    }

    emotion_map = {
        "01": "neutral",
        "02": "calm",
        "03": "happy",
        "04": "sad",
        "05": "angry",
        "06": "fearful",
        "07": "disgust",
        "08": "surprised"
    }

    intensity_map = {
        "01": "normal",
        "02": "strong"
    }

    statement_map = {
        "01": "kids are talking",
        "02": "dogs are sitting"
    }

    repetition_map = {
        "01": "first",
        "02": "second"
    }

    return {
        "modality": modality_map.get(parts[0]),
        "vocal_channel": vocal_channel_map.get(parts[1]),
        "emotion": emotion_map.get(parts[2]),
        "intensity": intensity_map.get(parts[3]),
        "statement": statement_map.get(parts[4]),
        "repetition": repetition_map.get(parts[5]),
        "actor": f"Actor_{parts[6]}"
    }

In [ ]:
parse_ravdess_filename("03-01-05-01-01-01-24.wav")

In [ ]:
#task 1 B

In [ ]:
emotion_rows = []

for path in wav_files:
    parts = path.stem.split("-")

    if len(parts) == 7:
        info = parse_ravdess_filename(str(path))
        emotion_rows.append(info["emotion"])

df_emotions = pd.DataFrame(emotion_rows, columns=["emotion"])

counts = df_emotions["emotion"].value_counts()


In [ ]:
plt.figure(figsize=(8, 5))
counts.plot(kind="bar")
plt.title("Emotion Distribution")
plt.xlabel("Emotion")
plt.ylabel("Number of Files")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
durations = []

for path in wav_files:
    parts = path.stem.split("-")

    if len(parts) == 7:
        try:
            y, sr = librosa.load(str(path), sr=None)
            duration = len(y) / sr
            durations.append(duration)
        except Exception as e:
            print("Skipped:", path.name, "| Error:", e)

durations = np.array(durations)



In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(durations, bins=20)
plt.title("Audio Duration Distribution")
plt.xlabel("Duration (seconds)")
plt.ylabel("Number of Files")
plt.tight_layout()
plt.show()

In [ ]:
samples = {
    "happy": None,
    "sad": None,
    "angry": None
}

for path in wav_files:
    parts = path.stem.split("-")

    if len(parts) == 7:
        info = parse_ravdess_filename(str(path))
        emotion = info["emotion"]

        if emotion in samples and samples[emotion] is None:
            samples[emotion] = path

print("Selected samples:")
for emotion, path in samples.items():
    print(emotion, "=>", path)

In [ ]:
fig, ax = plt.subplots(3, 2, figsize=(12, 10))

for i, (emotion, path) in enumerate(samples.items()):
    y, sr = librosa.load(str(path), sr=None)

    ax[i, 0].plot(y)
    ax[i, 0].set_title(f"{emotion} - Waveform")
    ax[i, 0].set_xlabel("Samples")
    ax[i, 0].set_ylabel("Amplitude")

    S = librosa.feature.melspectrogram(y=y, sr=sr)
    S_db = librosa.power_to_db(S, ref=np.max)

    librosa.display.specshow(
        S_db,
        sr=sr,
        x_axis="time",
        y_axis="mel",
        ax=ax[i, 1]
    )

    ax[i, 1].set_title(f"{emotion} - Mel Spectrogram")

plt.tight_layout()
plt.show()

In [ ]:
#task 2 A

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import opensmile
import seaborn as sns

In [ ]:
smile = opensmile.Smile(
    feature_set=opensmile.FeatureSet.eGeMAPSv02,
    feature_level=opensmile.FeatureLevel.Functionals,
)

In [ ]:
file_path = "data/ravdess/Actor_01/03-01-03-01-01-01-01.wav"

In [ ]:
features = smile.process_file(file_path)

In [ ]:
print(features.shape)

In [ ]:
print(features.columns.tolist())

In [ ]:
print(features.values)

In [ ]:
#task2 B

In [ ]:
import os
import pandas as pd
import opensmile

In [ ]:
smile = opensmile.Smile(
    feature_set=opensmile.FeatureSet.eGeMAPSv02,
    feature_level=opensmile.FeatureLevel.Functionals,
)

In [ ]:
DATASET_PATH = "data/ravdess"

In [ ]:
from tqdm import tqdm

rows = []

for root, _, files in os.walk(DATASET_PATH):
    for f in tqdm(files):
        if f.endswith(".wav"):
            path = os.path.join(root, f)
            
            feat = smile.process_file(path)
            emotion = f.split("-")[2]
            feat["label"] = emotion
            
            rows.append(feat)

In [ ]:
df = pd.concat(rows).reset_index(drop=True)

In [ ]:
print(df.shape)

In [ ]:
df.to_csv("ravdess_egemaps.csv", index=False)
print("Saved ✅")

In [ ]:
#task 2 C

In [ ]:
df = pd.read_csv("ravdess_egemaps.csv")

In [ ]:
pitch_feature = "F0semitoneFrom27.5Hz_sma3nz_amean"

mean_pitch = df.groupby("label")[pitch_feature].mean()

mean_pitch

In [ ]:
mean_pitch.sort_values().plot(kind="barh")
plt.title("Mean Pitch per Emotion")
plt.xlabel("Pitch (mean)")
plt.ylabel("Emotion")
plt.show()

In [ ]:
corr = df.drop("label", axis=1).corr()

In [ ]:
import seaborn as sns

plt.figure(figsize=(10, 8))
sns.heatmap(corr, cmap="coolwarm", center=0)
plt.title("Feature Correlation Heatmap")
plt.show()

In [ ]:
import numpy as np

corr_pairs = corr.abs().unstack()

# Remove self-correlations
corr_pairs = corr_pairs[corr_pairs < 1]

# Select strong ones
high_corr = corr_pairs[corr_pairs > 0.9]

high_corr.sort_values(ascending=False).head(10)

In [ ]:
#task3 A

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

In [ ]:
df = pd.read_csv("ravdess_egemaps.csv")

print(df.shape)
df.head()

In [ ]:
X = df.drop(columns=["label", "file"], errors="ignore")
y = df["label"]

X = X.fillna(0)

print("X shape:", X.shape)
print("y shape:", y.shape)
print(y.value_counts())

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
min_class_count = y.value_counts().min()

n_splits = min(10, min_class_count)

print("Using n_splits =", n_splits)

cv = StratifiedKFold(
    n_splits=n_splits,
    shuffle=True,
    random_state=42
)

In [ ]:
print("Dataset shape:", df.shape)
print(y.value_counts())
print("Minimum"), "samples per class:", y.value_counts().min()

In [ ]:
svm = SVC(kernel="rbf", C=1.0)

svm_acc_scores = cross_val_score(
    svm,
    X_scaled,
    y,
    cv=cv,
    scoring="accuracy"
)

svm_uar_scores = cross_val_score(
    svm,
    X_scaled,
    y,
    cv=cv,
    scoring="balanced_accuracy"
)

print("SVM Mean Accuracy:", svm_acc_scores.mean())
print("SVM Accuracy Std:", svm_acc_scores.std())

print("SVM Mean UAR:", svm_uar_scores.mean())
print("SVM UAR Std:", svm_uar_scores.std())

In [ ]:
svm_cv_pred = cross_val_predict(
    svm,
    X_scaled,
    y,
    cv=cv
)

labels = sorted(y.unique())

cm = confusion_matrix(
    y,
    svm_cv_pred,
    labels=labels
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=labels
)

fig, ax = plt.subplots(figsize=(8, 6))
disp.plot(cmap="Blues", xticks_rotation=45, ax=ax)
plt.title("SVM Confusion Matrix")
plt.tight_layout()
plt.show()


In [ ]:
svm_c_results = []

for C_val in [0.1, 1, 10]:
    svm_temp = SVC(kernel="rbf", C=C_val)

    scores = cross_val_score(
        svm_temp,
        X_scaled,
        y,
        cv=cv,
        scoring="balanced_accuracy"
    )

    svm_c_results.append({
        "C": C_val,
        "Mean UAR": scores.mean(),
        "UAR Std": scores.std()
    })

svm_c_results_df = pd.DataFrame(svm_c_results)
svm_c_results_df

In [ ]:
#task 3 B

In [ ]:
rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

rf_acc_scores = cross_val_score(
    rf,
    X,
    y,
    cv=cv,
    scoring="accuracy"
)

rf_uar_scores = cross_val_score(
    rf,
    X,
    y,
    cv=cv,
    scoring="balanced_accuracy"
)

print("Random Forest Mean Accuracy:", rf_acc_scores.mean())
print("Random Forest Accuracy Std:", rf_acc_scores.std())

print("Random Forest Mean UAR:", rf_uar_scores.mean())
print("Random Forest UAR Std:", rf_uar_scores.std())

In [ ]:
rf.fit(X, y)

feature_importance = pd.Series(
    rf.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

feature_importance.head(20)

In [ ]:
def feature_group(feature_name):
    name = feature_name.lower()

    if "f0" in name or "pitch" in name:
        return "pitch"
    elif "loudness" in name or "energy" in name or "rms" in name:
        return "energy"
    elif "mfcc" in name:
        return "mfcc"
    elif "spectral" in name or "alpha" in name or "hammarberg" in name:
        return "spectral"
    else:
        return "other"


color_map = {
    "pitch": "tab:blue",
    "energy": "tab:orange",
    "mfcc": "tab:green",
    "spectral": "tab:red",
    "other": "tab:gray"
}

In [ ]:
def feature_group(feature_name):
    name = feature_name.lower()

    if "f0" in name or "pitch" in name:
        return "pitch"
    elif "loudness" in name or "energy" in name or "rms" in name:
        return "energy"
    elif "mfcc" in name:
        return "mfcc"
    elif "spectral" in name or "alpha" in name or "hammarberg" in name:
        return "spectral"
    else:
        return "other"


color_map = {
    "pitch": "tab:blue",
    "energy": "tab:orange",
    "mfcc": "tab:green",
    "spectral": "tab:red",
    "other": "tab:gray"
}

In [ ]:
import matplotlib.patches as mpatches

legend_handles = [
    mpatches.Patch(color=color, label=group)
    for group, color in color_map.items()
]

plt.figure(figsize=(6, 2))
plt.legend(handles=legend_handles, loc="center", ncol=3)
plt.axis("off")
plt.title("Feature Group Colour Legend")
plt.show()


In [ ]:
#task 3 C

In [ ]:
comparison = pd.DataFrame({
    "Model": ["SVM (RBF)", "Random Forest"],
    "Features": ["eGeMAPS", "eGeMAPS"],
    "Mean Acc.": [svm_acc_scores.mean(), rf_acc_scores.mean()],
    "Mean UAR": [svm_uar_scores.mean(), rf_uar_scores.mean()],
    "UAR Std": [svm_uar_scores.std(), rf_uar_scores.std()]
})

comparison

In [ ]:
comparison_rounded = comparison.copy()

for col in ["Mean Acc.", "Mean UAR", "UAR Std"]:
    comparison_rounded[col] = comparison_rounded[col].round(4)

comparison_rounded